# SANPO YOLO Gap Analysis Notebook

**Purpose:** Identify which scene objects in SANPO are detected by **depth** but are completely missed by **YOLO26n** using its current COCO class whitelist.

The output of this notebook is:
1. **Per-frame visualizations** (RGB + bounding boxes + depth heatmap + unlabeled blob outlines)
2. **A frequency count** of how large/common the depth-only regions are — this tells us which classes to prioritize in YOLO fine-tuning
3. **A summary CSV** with per-frame gap statistics

---
**Data structure assumed:**
```
data/sanpo/raw/<session_hash>/camera_head/left/video_frames/000000.png
data/sanpo/raw/<session_hash>/camera_head/left/depth_maps/000000.float16.gz
```

In [ ]:
# ── Cell 1: Config ─────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to path so we can import src modules directly
REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

# ── Data Paths ─────────────────────────────────────────────────────────────────
SANPO_RAW = REPO_ROOT / "data" / "sanpo" / "raw"

# Pick which session(s) to analyse. Set to None to run all sessions.
SESSION_FILTER = None   # e.g. ["01cbb9d5..."] to restrict

# How many frames to sample per session (None = all frames)
MAX_FRAMES_PER_SESSION = 30

# Depth alert range (metres). Blobs within this range are considered candidate hazards.
DEPTH_MIN_M = 0.5
DEPTH_MAX_M = 6.0

# Min blob pixel area to count as an unlabeled obstruction
MIN_BLOB_AREA_PX = 800

# Grid columns for the unlabeled sweep (same as pipeline.py)
OBSTACLE_GRID_COLS = 5

# YOLO model path
YOLO_MODEL_PATH = str(REPO_ROOT / "yolo26n.pt")

# Output dir for saved visualizations
OUTPUT_DIR = REPO_ROOT / "notebooks" / "gap_analysis_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ SANPO root:   {SANPO_RAW}")
print(f"✅ YOLO model:   {YOLO_MODEL_PATH}")
print(f"✅ Output dir:   {OUTPUT_DIR}")

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import gzip
import json
import csv
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import ndimage
from ultralytics import YOLO

print("✅ All imports OK")

In [ ]:
# ── Cell 3: Depth Loader (matches depth_loader.py behaviour) ───────────────────

def load_sanpo_depth(depth_path: Path) -> np.ndarray | None:
    """
    Loads a SANPO .float16.gz depth file.
    Returns a float32 depth array in metres, or None on failure.
    Mirrors src/perception_stack/depth_loader.py behaviour.
    """
    try:
        with gzip.open(depth_path, 'rb') as f:
            raw = np.frombuffer(f.read(), dtype=np.float16)
        # SANPO has 2 padding values at the start → trim them
        raw = raw[2:]
        # Attempt to reconstruct shape from rgb frame (1242 x 2208 for ZED)
        # Try common SANPO shapes
        for h, w in [(1242, 2208), (720, 1280), (1080, 1920)]:
            if raw.size == h * w:
                return raw.reshape(h, w).astype(np.float32)
        # Fallback: try to infer shape
        side = int(np.sqrt(raw.size))
        return raw[:side*side].reshape(side, side).astype(np.float32)
    except Exception as e:
        print(f"  ⚠️ Depth load failed: {e}")
        return None


def get_sessions(sanpo_raw: Path, session_filter=None):
    """Return list of session paths."""
    sessions = sorted([p for p in sanpo_raw.iterdir() if p.is_dir()])
    if session_filter:
        sessions = [s for s in sessions if s.name in session_filter]
    return sessions


def get_frame_paths(session_path: Path):
    """Return sorted list of (rgb_path, depth_path) tuples for a session."""
    cam_dir = session_path / "camera_head" / "left"
    rgb_dir = cam_dir / "video_frames"
    dep_dir = cam_dir / "depth_maps"

    rgb_files = sorted(rgb_dir.glob("*.png"))
    pairs = []
    for rgb_f in rgb_files:
        stem = rgb_f.stem
        dep_f = dep_dir / f"{stem}.float16.gz"
        if dep_f.exists():
            pairs.append((rgb_f, dep_f))
    return pairs


# Sanity check on first session
sessions = get_sessions(SANPO_RAW, SESSION_FILTER)
print(f"Found {len(sessions)} session(s): {[s.name[:12]+'...' for s in sessions]}")
first_pairs = get_frame_paths(sessions[0])
print(f"Session 0 has {len(first_pairs)} frame pairs")

In [ ]:
# ── Cell 4: YOLO Setup ─────────────────────────────────────────────────────────

# These are the CURRENT allowed classes (from yolo_tracker.py)
ALLOWED_CLASSES = {
    "person", "bicycle", "car", "motorcycle", "bus", "truck",
    "dog", "cat", "traffic light", "stop sign", "umbrella",
    "backpack", "suitcase",
}

model = YOLO(YOLO_MODEL_PATH)
print(f"✅ YOLO model loaded: {YOLO_MODEL_PATH}")

def run_yolo(frame_bgr: np.ndarray) -> list[dict]:
    """Run YOLO on a frame, return list of detection dicts (filtered by ALLOWED_CLASSES)."""
    results = model.predict(frame_bgr, conf=0.30, verbose=False)[0]
    dets = []
    if results.boxes is None:
        return dets
    for box, cls_idx, conf in zip(
        results.boxes.xyxy.cpu().numpy(),
        results.boxes.cls.cpu().numpy().astype(int),
        results.boxes.conf.cpu().numpy(),
    ):
        cls_name = results.names[cls_idx]
        if cls_name not in ALLOWED_CLASSES:
            continue
        x1, y1, x2, y2 = map(int, box)
        dets.append({
            "class_name": cls_name,
            "conf": round(float(conf), 3),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
        })
    return dets

In [ ]:
# ── Cell 5: Depth Gap Detector ─────────────────────────────────────────────────

def find_depth_gaps(depth_map: np.ndarray,
                    yolo_dets: list[dict],
                    frame_h: int, frame_w: int) -> list[dict]:
    """
    Find depth regions within DEPTH_MIN_M..DEPTH_MAX_M that have no
    corresponding YOLO bounding box. These are the 'gaps'.

    Returns a list of blob dicts:
        cx, cy     : centroid of the blob in depth-map pixel coords
        area_px    : pixel area of the blob
        min_depth  : minimum depth in blob (metres)
        mean_depth : mean depth in blob (metres)
        elevation  : 'head_level' | 'mid_level' | 'foot_level' (relative to frame_h)
        col_idx    : which of the 5 grid columns it's in
    """
    dh, dw = depth_map.shape

    # 1. Create a binary mask: True where depth is in alert range
    alert_mask = (depth_map >= DEPTH_MIN_M) & (depth_map <= DEPTH_MAX_M)

    # 2. Mask out pixels already covered by YOLO bounding boxes
    #    (scale YOLO coords to depth map coords if sizes differ)
    scale_x = dw / frame_w
    scale_y = dh / frame_h
    for det in yolo_dets:
        dx1 = int(det["x1"] * scale_x)
        dy1 = int(det["y1"] * scale_y)
        dx2 = int(det["x2"] * scale_x)
        dy2 = int(det["y2"] * scale_y)
        alert_mask[dy1:dy2, dx1:dx2] = False

    # 3. Focus on lower 60% of the depth map (navigation-relevant zone)
    nav_start_row = int(dh * 0.4)
    nav_mask = np.zeros_like(alert_mask)
    nav_mask[nav_start_row:, :] = alert_mask[nav_start_row:, :]

    # 4. Connected component labeling
    labeled, num_features = ndimage.label(nav_mask)

    blobs = []
    for lbl in range(1, num_features + 1):
        blob_pixels = np.where(labeled == lbl)
        area = len(blob_pixels[0])
        if area < MIN_BLOB_AREA_PX:
            continue

        cy_dm = int(np.mean(blob_pixels[0]))
        cx_dm = int(np.mean(blob_pixels[1]))
        depths = depth_map[blob_pixels]
        depths = depths[(depths >= DEPTH_MIN_M) & (depths <= DEPTH_MAX_M)]

        if len(depths) == 0:
            continue

        # Elevation classification (relative to full frame height)
        rel_y = cy_dm / dh
        if rel_y < 0.45:
            elevation = "head_level"
        elif rel_y < 0.65:
            elevation = "mid_level"
        else:
            elevation = "foot_level"

        # Column classification (0-4)
        col_idx = min(int(cx_dm / dw * OBSTACLE_GRID_COLS), OBSTACLE_GRID_COLS - 1)

        blobs.append({
            "cx_dm": cx_dm, "cy_dm": cy_dm,
            # Convert back to rgb frame coords for visualization
            "cx_rgb": int(cx_dm / scale_x),
            "cy_rgb": int(cy_dm / scale_y),
            "area_px": area,
            "min_depth": float(np.min(depths)),
            "mean_depth": float(np.mean(depths)),
            "elevation": elevation,
            "col_idx": col_idx,
        })

    return sorted(blobs, key=lambda b: b["mean_depth"])

In [ ]:
# ── Cell 6: Visualization Helper ───────────────────────────────────────────────

ELEVATION_COLORS = {
    "head_level": (255, 50,  50),   # Red   — most dangerous
    "mid_level":  (255, 165,  0),   # Orange
    "foot_level": (50,  200, 50),   # Green — trip hazard
}
YOLO_BOX_COLOR  = (100, 200, 255)  # Light blue for YOLO boxes
YOLO_TEXT_COLOR = (0,   0,   0)


def visualize_frame(rgb_path: Path,
                    depth_map: np.ndarray,
                    yolo_dets: list[dict],
                    depth_blobs: list[dict],
                    save_path: Path | None = None,
                    show: bool = True):
    """
    3-panel visualization:
      Left  : RGB with YOLO boxes (blue) + depth-gap centroids (colored circles)
      Center: Depth heatmap (clipped to 0-10m)
      Right  : Depth gap mask only
    """
    rgb_bgr = cv2.imread(str(rgb_path))
    if rgb_bgr is None:
        print(f"  ⚠️ Could not read {rgb_path}")
        return
    frame_h, frame_w = rgb_bgr.shape[:2]
    rgb_vis = rgb_bgr.copy()

    # Draw YOLO detections (blue)
    for det in yolo_dets:
        cv2.rectangle(rgb_vis,
                      (det["x1"], det["y1"]), (det["x2"], det["y2"]),
                      YOLO_BOX_COLOR, 2)
        label = f"{det['class_name']} {det['conf']:.2f}"
        cv2.putText(rgb_vis, label,
                    (det["x1"], max(det["y1"]-6, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, YOLO_BOX_COLOR, 2)

    # Draw depth gap blobs (colored circles by elevation)
    for blob in depth_blobs:
        color = ELEVATION_COLORS.get(blob["elevation"], (200, 200, 200))
        radius = max(10, int(np.sqrt(blob["area_px"]) * 0.15))
        cv2.circle(rgb_vis,
                   (blob["cx_rgb"], blob["cy_rgb"]),
                   radius, color, -1)
        cv2.circle(rgb_vis,
                   (blob["cx_rgb"], blob["cy_rgb"]),
                   radius, (255,255,255), 2)
        label = f"{blob['elevation'][:4]} {blob['mean_depth']:.1f}m"
        cv2.putText(rgb_vis, label,
                    (blob["cx_rgb"]+radius+4, blob["cy_rgb"]+5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)

    # Depth heatmap (normalize to 0-10m)
    depth_display = np.clip(depth_map, 0, 10.0)
    depth_norm = (depth_display / 10.0 * 255).astype(np.uint8)
    depth_color = cv2.applyColorMap(depth_norm, cv2.COLORMAP_TURBO)
    # Resize to match RGB height for side-by-side
    depth_color_r = cv2.resize(depth_color, (frame_w, frame_h))

    # Gap-only mask (show nav zone only)
    scale_x = depth_map.shape[1] / frame_w
    scale_y = depth_map.shape[0] / frame_h
    gap_mask = np.zeros((frame_h, frame_w, 3), dtype=np.uint8)
    for blob in depth_blobs:
        color = ELEVATION_COLORS.get(blob["elevation"], (200, 200, 200))
        radius = max(12, int(np.sqrt(blob["area_px"]) * 0.12))
        cv2.circle(gap_mask, (blob["cx_rgb"], blob["cy_rgb"]), radius, color, -1)

    # Compose side-by-side
    panel = np.hstack([rgb_vis, depth_color_r, gap_mask])
    panel_rgb = cv2.cvtColor(panel, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 1, figsize=(20, 6))
    ax.imshow(panel_rgb)
    ax.axis("off")

    # Legend
    patches = [
        mpatches.Patch(color=np.array(YOLO_BOX_COLOR)/255, label="YOLO detection"),
        mpatches.Patch(color=np.array(ELEVATION_COLORS["head_level"])/255, label="Gap: head level"),
        mpatches.Patch(color=np.array(ELEVATION_COLORS["mid_level"])/255, label="Gap: mid level"),
        mpatches.Patch(color=np.array(ELEVATION_COLORS["foot_level"])/255, label="Gap: foot level"),
    ]
    ax.legend(handles=patches, loc="upper right", fontsize=9,
              framealpha=0.85, ncol=2)

    n_yolo = len(yolo_dets)
    n_gap  = len(depth_blobs)
    ax.set_title(
        f"{rgb_path.name} | YOLO detections: {n_yolo}  |  "
        f"Depth-only gaps (unlabelled): {n_gap}  |  "
        f"[Left: RGB+YOLO+gaps]  [Centre: Depth heatmap 0-10m]  [Right: Gap-only]",
        fontsize=10, pad=8
    )
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()

print("✅ Visualization helper ready")

In [ ]:
# ── Cell 7: Main Analysis Loop ─────────────────────────────────────────────────

all_stats = []  # Will be dumped to CSV at the end

for session in sessions:
    pairs = get_frame_paths(session)
    if MAX_FRAMES_PER_SESSION:
        # Sample evenly across the session rather than just the first N frames
        step = max(1, len(pairs) // MAX_FRAMES_PER_SESSION)
        pairs = pairs[::step][:MAX_FRAMES_PER_SESSION]

    print(f"\n📂 Session: {session.name[:20]}... ({len(pairs)} frames)")

    for rgb_path, dep_path in pairs:
        stem = rgb_path.stem

        # Load RGB
        frame_bgr = cv2.imread(str(rgb_path))
        if frame_bgr is None:
            continue
        fh, fw = frame_bgr.shape[:2]

        # Load depth
        depth = load_sanpo_depth(dep_path)
        if depth is None:
            continue

        # YOLO inference
        yolo_dets = run_yolo(frame_bgr)

        # Find depth-only gaps (unlabelled obstructions)
        blobs = find_depth_gaps(depth, yolo_dets, fh, fw)

        # Per-frame stats
        row = {
            "session": session.name[:20],
            "frame":   stem,
            "yolo_detections": len(yolo_dets),
            "yolo_classes": "|".join(sorted(set(d["class_name"] for d in yolo_dets))),
            "depth_gaps_total": len(blobs),
            "gaps_head_level":  sum(1 for b in blobs if b["elevation"] == "head_level"),
            "gaps_mid_level":   sum(1 for b in blobs if b["elevation"] == "mid_level"),
            "gaps_foot_level":  sum(1 for b in blobs if b["elevation"] == "foot_level"),
            "nearest_gap_m":    round(min((b["min_depth"] for b in blobs), default=99.0), 2),
            "largest_gap_area": max((b["area_px"] for b in blobs), default=0),
        }
        all_stats.append(row)

        # Only visualize frames that HAVE gaps  →  these are your interesting cases
        has_gaps = len(blobs) > 0
        save_path = OUTPUT_DIR / f"{session.name[:12]}_{stem}.png" if has_gaps else None

        print(f"  Frame {stem}: YOLO={len(yolo_dets)}  Gaps={len(blobs)}"
              + (f"  (nearest {row['nearest_gap_m']}m)" if blobs else ""))

        if has_gaps:
            visualize_frame(rgb_path, depth, yolo_dets, blobs,
                            save_path=save_path, show=True)

print("\n✅ Analysis loop complete")

In [ ]:
# ── Cell 8: Summary Statistics ─────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_stats)

print("\n═══════════════════════════ SUMMARY ═══════════════════════════")
print(f"Total frames analysed:          {len(df)}")
print(f"Frames with ZERO YOLO hits:     {(df.yolo_detections == 0).sum()}")
print(f"Frames with depth gaps:         {(df.depth_gaps_total > 0).sum()}")
print(f"Frames with head-level gaps:    {(df.gaps_head_level > 0).sum()}")
print(f"Frames with foot-level gaps:    {(df.gaps_foot_level > 0).sum()}")
print()
print("── YOLO class frequency (what YOLO IS catching) ──")
all_classes = [c for row in df.yolo_classes for c in row.split("|") if c]
from collections import Counter
for cls, count in Counter(all_classes).most_common():
    print(f"  {cls:<25} {count:>4} frames")

print()
print("── Depth gap distribution by elevation ──")
print(f"  Head level gaps (face/chest):  {df.gaps_head_level.sum():>4}")
print(f"  Mid level gaps (waist):        {df.gaps_mid_level.sum():>4}")
print(f"  Foot level gaps (ground):      {df.gaps_foot_level.sum():>4}")

print()
print("── Top 10 frames by number of depth gaps ──")
print(df.sort_values("depth_gaps_total", ascending=False)[
    ["session", "frame", "yolo_detections", "depth_gaps_total",
     "nearest_gap_m", "largest_gap_area"]
].head(10).to_string(index=False))

In [ ]:
# ── Cell 9: Export Stats to CSV ────────────────────────────────────────────────
csv_path = OUTPUT_DIR / "gap_analysis_stats.csv"
df.to_csv(csv_path, index=False)
print(f"✅ Stats saved to: {csv_path}")
df.head(10)

In [ ]:
# ── Cell 10: Manually inspect any single frame ─────────────────────────────────
# Set the paths below to look at any specific frame

SESSION_NAME  = list(get_sessions(SANPO_RAW))[0].name
FRAME_INDEX   = "000050"   # ← change this

session_path  = SANPO_RAW / SESSION_NAME
rgb_path      = session_path / "camera_head" / "left" / "video_frames" / f"{FRAME_INDEX}.png"
dep_path      = session_path / "camera_head" / "left" / "depth_maps"   / f"{FRAME_INDEX}.float16.gz"

frame_bgr = cv2.imread(str(rgb_path))
depth     = load_sanpo_depth(dep_path)
yolo_dets = run_yolo(frame_bgr)
blobs     = find_depth_gaps(depth, yolo_dets, *frame_bgr.shape[:2])

print(f"Frame {FRAME_INDEX}: {len(yolo_dets)} YOLO detections, {len(blobs)} depth gaps")
for b in blobs:
    print(f"  Gap @ ({b['cx_rgb']}, {b['cy_rgb']})  "
          f"depth={b['mean_depth']:.2f}m  elevation={b['elevation']}  area={b['area_px']}px")

visualize_frame(rgb_path, depth, yolo_dets, blobs, show=True)